# Simulacion del Experimento de la Doble Rendija
## Difraccion de Fraunhofer — Python con matplotlib

Este notebook simula el patron de interferencia de la doble rendija usando la teoria ondulatoria clasica.

**Formula central (Fraunhofer):**

Para doble rendija:
$$I(\\theta) = I_0 \\cdot \\left(\\frac{\\sin(\\alpha)}{\\alpha}\\right)^2 \\cdot \\cos^2(\\beta)$$

Donde:
- $\\alpha = \\frac{\\pi a \\sin\\theta}{\\lambda}$ — controla el ancho del lobulo central (ancho de rendija `a`)
- $\\beta = \\frac{\\pi d \\sin\\theta}{\\lambda}$ — controla las franjas de interferencia (separacion `d`)
- `a` = ancho de cada rendija
- `d` = separacion entre centros de rendijas
- `lambda` = longitud de onda del laser
- `L` = distancia a la pantalla


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 1 — Importaciones
# ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

# Estilo oscuro que imita la pantalla del laboratorio
plt.rcParams.update({
    'figure.facecolor': '#0a0a12',
    'axes.facecolor':   '#0a0a12',
    'axes.edgecolor':   '#3a3a4a',
    'axes.labelcolor':  '#c8c0b0',
    'xtick.color':      '#8a8a9a',
    'ytick.color':      '#8a8a9a',
    'text.color':       '#c8c0b0',
    'grid.color':       '#2a2a3a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
})

print('Librerias cargadas correctamente')
print(f'  numpy      {np.__version__}')
print(f'  matplotlib {plt.matplotlib.__version__}')


## Celda 2 — Funciones fisicas

Aqui definimos toda la fisica del experimento:
- Conversion de longitud de onda a color RGB
- Calculo de intensidad (Fraunhofer)
- Posiciones en la pantalla a partir del angulo


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 2 — Funciones fisicas
# ─────────────────────────────────────────────────────────────

def longitud_onda_a_rgb(lam_nm):
    """
    Convierte una longitud de onda visible (380-780 nm) a color RGB
    normalizado [0,1]. Usa el modelo perceptual estandar con
    correccion gamma para reproducir el color real del laser.
    """
    wl = lam_nm
    if   380 <= wl < 440: r, g, b = (440-wl)/60,        0.0,         1.0
    elif 440 <= wl < 490: r, g, b = 0.0,          (wl-440)/50,       1.0
    elif 490 <= wl < 510: r, g, b = 0.0,          1.0,           (510-wl)/20
    elif 510 <= wl < 580: r, g, b = (wl-510)/70,  1.0,               0.0
    elif 580 <= wl < 645: r, g, b = 1.0,          (645-wl)/65,       0.0
    else:                 r, g, b = 1.0,          0.0,               0.0

    # Factor de visibilidad (extremos del espectro parecen mas tenues)
    if   380 <= wl < 420: factor = 0.3 + 0.7*(wl-380)/40
    elif 420 <= wl <= 700: factor = 1.0
    else:                  factor = 0.3 + 0.7*(780-wl)/80

    gamma = 0.8
    return (r*factor)**gamma, (g*factor)**gamma, (b*factor)**gamma


def intensidad_fraunhofer(y_mm, a_mm, d_mm, lam_mm, L_mm, modo='doble'):
    """
    Calcula la intensidad I(y) normalizada usando difraccion de Fraunhofer.

    Parametros:
        y_mm  : posiciones en la pantalla [mm] — array numpy
        a_mm  : ancho de cada rendija [mm]
        d_mm  : separacion entre centros de rendijas [mm]
        lam_mm: longitud de onda [mm]
        L_mm  : distancia rendija-pantalla [mm]
        modo  : 'doble' o 'simple'

    Retorna: array de intensidades en [0, 1]
    """
    # sin(theta) exacto — sin aproximacion paraxial
    sin_theta = y_mm / np.sqrt(y_mm**2 + L_mm**2)

    # Parametro de difraccion alpha (apertura individual)
    # alpha = pi * a * sin(theta) / lambda
    alpha = np.pi * a_mm * sin_theta / lam_mm

    # sinc^2 — envolvente de difraccion por apertura
    # Usamos where para manejar alpha=0 sin division por cero
    sinc = np.where(np.abs(alpha) < 1e-12, 1.0, np.sin(alpha)/alpha)
    I_difraccion = sinc**2

    if modo == 'doble':
        # Parametro de interferencia beta (separacion entre rendijas)
        # beta = pi * d * sin(theta) / lambda
        beta = np.pi * d_mm * sin_theta / lam_mm
        # cos^2(beta) — factor de interferencia entre las dos rendijas
        I_interferencia = np.cos(beta)**2
        return I_difraccion * I_interferencia
    else:
        # Rendija simple: solo difraccion, sin interferencia
        return I_difraccion


def crear_colormap_laser(lam_nm):
    """Crea un colormap que va de negro al color del laser."""
    r, g, b = longitud_onda_a_rgb(lam_nm)
    cmap = LinearSegmentedColormap.from_list(
        f'laser_{lam_nm}nm',
        [(0, (0, 0, 0)), (1, (r, g, b))]
    )
    return cmap, (r, g, b)


# --- Verificacion rapida ---
print('Funciones fisicas definidas')
r, g, b = longitud_onda_a_rgb(650)
print(f'  Color laser rojo 650nm: RGB = ({r:.3f}, {g:.3f}, {b:.3f})')
r2, g2, b2 = longitud_onda_a_rgb(532)
print(f'  Color laser verde 532nm: RGB = ({r2:.3f}, {g2:.3f}, {b2:.3f})')


## Celda 3 — Parametros del experimento

Modifica estos valores para reproducir tus observaciones del laboratorio.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 3 — Parametros del experimento (MODIFICA AQUI)
# ─────────────────────────────────────────────────────────────

# --- Laser ---
LAMBDA_NM = 650          # longitud de onda [nm] — rojo tipico de diodo
LAMBDA_MM = LAMBDA_NM * 1e-6   # convertido a mm (1 nm = 1e-6 mm)

# --- Geometria de las rendijas ---
A_MM = 0.04   # ancho de cada rendija [mm]
D_MM = 0.25   # separacion entre centros de rendijas [mm]

# --- Montaje experimental ---
L_CM = 100    # distancia rendija-pantalla [cm]
L_MM = L_CM * 10   # convertido a mm

# --- Pantalla ---
Y_RANGO_MM = 15    # mitad del rango de la pantalla [mm]
N_PUNTOS   = 4000  # resolucion del calculo

# Vector de posiciones
y = np.linspace(-Y_RANGO_MM, Y_RANGO_MM, N_PUNTOS)

print('Parametros actuales:')
print(f'  Longitud de onda : lambda = {LAMBDA_NM} nm')
print(f'  Ancho rendija    : a = {A_MM} mm')
print(f'  Separacion       : d = {D_MM} mm')
print(f'  Dist. pantalla   : L = {L_CM} cm')
print()

# Predicciones teoricas
delta_y = LAMBDA_MM * L_MM / D_MM
lobulo  = 2 * LAMBDA_MM * L_MM / A_MM
print('Predicciones teoricas:')
print(f'  Separacion entre franjas  : delta_y = lambda*L/d = {delta_y:.2f} mm')
print(f'  Ancho del lobulo central  : 2*lambda*L/a = {lobulo:.1f} mm')
print(f'  Franjas dentro del lobulo : aprox. {lobulo/delta_y:.0f}')


## Celda 4 — Patron de interferencia principal

Reproducimos lo que se ve en la pantalla del laboratorio:
- La franja superior es la imagen visual del patron de luz
- La grafica inferior es la curva de intensidad I(y)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 4 — Grafica principal del patron
# ─────────────────────────────────────────────────────────────

# Calcular intensidades
I_doble  = intensidad_fraunhofer(y, A_MM, D_MM, LAMBDA_MM, L_MM, modo='doble')
I_simple = intensidad_fraunhofer(y, A_MM, D_MM, LAMBDA_MM, L_MM, modo='simple')

cmap_laser, color_rgb = crear_colormap_laser(LAMBDA_NM)

fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(3, 1, height_ratios=[1, 0.08, 1], hspace=0.05)

# ── Panel 1: imagen visual del patron ──
ax_img = fig.add_subplot(gs[0])
imagen = np.sqrt(I_doble).reshape(1, -1)   # sqrt -> mas contraste visual
ax_img.imshow(
    np.tile(imagen, (80, 1)),
    aspect='auto',
    extent=[-Y_RANGO_MM, Y_RANGO_MM, 0, 1],
    cmap=cmap_laser, vmin=0, vmax=1, origin='lower'
)
ax_img.set_yticks([])
ax_img.set_xticks([])
ax_img.set_title(
    f'Patron de interferencia — doble rendija\n'
    f'lambda={LAMBDA_NM} nm   a={A_MM} mm   d={D_MM} mm   L={L_CM} cm',
    pad=10, fontsize=11
)

# ── Separador invisible ──
ax_sep = fig.add_subplot(gs[1])
ax_sep.axis('off')

# ── Panel 2: curva de intensidad ──
ax_c = fig.add_subplot(gs[2])
ax_c.plot(y, I_doble,  color=color_rgb, lw=1.5, label='Doble rendija I(y)')
ax_c.plot(y, I_simple, color='white',   lw=0.8, alpha=0.35,
          linestyle='--', label='Envolvente difraccion (simple)')
ax_c.fill_between(y, I_doble, alpha=0.15, color=color_rgb)
ax_c.set_xlabel('Posicion en pantalla y [mm]', fontsize=10)
ax_c.set_ylabel('Intensidad I / I0', fontsize=10)
ax_c.set_xlim(-Y_RANGO_MM, Y_RANGO_MM)
ax_c.set_ylim(-0.05, 1.1)
ax_c.grid(True)
ax_c.legend(fontsize=9, loc='upper right')

# Marcar separacion teorica entre franjas
delta_y_plot = LAMBDA_MM * L_MM / D_MM
ax_c.axvline( delta_y_plot, color='yellow', lw=0.6, alpha=0.4, linestyle=':')
ax_c.axvline(-delta_y_plot, color='yellow', lw=0.6, alpha=0.4, linestyle=':')
ax_c.text(delta_y_plot + 0.3, 0.9, f'dy={delta_y_plot:.1f}mm',
          fontsize=8, color='yellow', alpha=0.8)

plt.savefig('patron_doble_rendija.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a12')
plt.show()
print('Figura guardada como patron_doble_rendija.png')


## Celda 5 — Comparacion: simple vs doble rendija

Reproducimos la diferencia clave observada en el laboratorio:
- **Simple**: lobulo central ancho, sin franjas finas
- **Doble**: franjas finas de interferencia dentro del lobulo de difraccion


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 5 — Comparacion simple vs doble rendija
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 8),
                         gridspec_kw={'height_ratios': [1, 2.5]})
fig.suptitle('Comparacion: rendija simple vs doble rendija', fontsize=12)

for col, (modo, titulo) in enumerate([
    ('simple', f'Rendija SIMPLE\na={A_MM} mm'),
    ('doble',  f'Rendija DOBLE\na={A_MM} mm, d={D_MM} mm')
]):
    I = intensidad_fraunhofer(y, A_MM, D_MM, LAMBDA_MM, L_MM, modo=modo)
    imagen = np.sqrt(I).reshape(1, -1)

    # Imagen visual
    ax_img = axes[0][col]
    ax_img.imshow(
        np.tile(imagen, (60, 1)),
        aspect='auto',
        extent=[-Y_RANGO_MM, Y_RANGO_MM, 0, 1],
        cmap=cmap_laser, vmin=0, vmax=1, origin='lower'
    )
    ax_img.set_yticks([])
    ax_img.set_xticks([])
    ax_img.set_title(titulo, fontsize=10)

    # Curva de intensidad
    ax_curva = axes[1][col]
    ax_curva.plot(y, I, color=color_rgb, lw=1.5)
    ax_curva.fill_between(y, I, alpha=0.15, color=color_rgb)
    ax_curva.set_xlabel('Posicion en pantalla y [mm]', fontsize=9)
    ax_curva.set_ylabel('Intensidad I / I0', fontsize=9)
    ax_curva.set_xlim(-Y_RANGO_MM, Y_RANGO_MM)
    ax_curva.set_ylim(-0.05, 1.1)
    ax_curva.grid(True)

    # Anotaciones explicativas
    dy = LAMBDA_MM * L_MM / D_MM
    if modo == 'simple':
        ax_curva.annotate('Lobulo central\nancho (solo difraccion)',
                          xy=(0, 1), xytext=(4, 0.75),
                          arrowprops=dict(arrowstyle='->', color='white', lw=0.8),
                          color='white', fontsize=8)
    else:
        ax_curva.annotate(f'Franjas finas\ndy={dy:.1f}mm',
                          xy=(dy, 1), xytext=(4, 0.75),
                          arrowprops=dict(arrowstyle='->', color='white', lw=0.8),
                          color='white', fontsize=8)

plt.tight_layout()
plt.savefig('comparacion_rendijas.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a12')
plt.show()
print('Figura guardada como comparacion_rendijas.png')


## Celda 6 — Efecto de los parametros (tus observaciones del lab)

Reproducimos exactamente lo que observaste al variar **a** y **d**:

| Configuracion | Observacion |
|---|---|
| a=0.04, d=0.25 | Picos y sombras muy notorios |
| a=0.04, d=0.50 | Franjas mas juntas, dificiles de distinguir |
| a=0.08, d=0.25 | Lobulo central mas angosto |
| a=0.08, d=0.50 | Mas angosto y mas franjas |


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 6 — Exploracion de parametros (observaciones del lab)
# ─────────────────────────────────────────────────────────────

configuraciones = [
    (0.04, 0.25, 'a=0.04, d=0.25\npicos muy notorios'),
    (0.04, 0.50, 'a=0.04, d=0.50\nfranjas comprimidas'),
    (0.08, 0.25, 'a=0.08, d=0.25\nlobulo mas angosto'),
    (0.08, 0.50, 'a=0.08, d=0.50\nmas franjas y angosto'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 6),
                         gridspec_kw={'height_ratios': [1, 2.5]})
fig.suptitle('Efecto de variar a (ancho) y d (separacion)', fontsize=12)

for col, (a, d, etiqueta) in enumerate(configuraciones):
    I = intensidad_fraunhofer(y, a, d, LAMBDA_MM, L_MM, modo='doble')
    imagen = np.sqrt(I).reshape(1, -1)

    # Imagen visual
    ax_img = axes[0][col]
    ax_img.imshow(np.tile(imagen, (50, 1)), aspect='auto',
                  extent=[-Y_RANGO_MM, Y_RANGO_MM, 0, 1],
                  cmap=cmap_laser, vmin=0, vmax=1, origin='lower')
    ax_img.set_yticks([])
    ax_img.set_xticks([])
    ax_img.set_title(etiqueta, fontsize=8)

    # Curva de intensidad
    ax_curva = axes[1][col]
    ax_curva.plot(y, I, color=color_rgb, lw=1.2)
    ax_curva.fill_between(y, I, alpha=0.12, color=color_rgb)
    ax_curva.set_xlim(-12, 12)
    ax_curva.set_ylim(-0.05, 1.1)
    ax_curva.set_xlabel('y [mm]', fontsize=8)
    if col == 0:
        ax_curva.set_ylabel('I / I0', fontsize=8)
    ax_curva.grid(True)
    ax_curva.tick_params(labelsize=7)

    # Anotar separacion teorica entre franjas
    dy = LAMBDA_MM * L_MM / d
    ax_curva.text(0.05, 0.92, f'dy={dy:.1f}mm',
                 transform=ax_curva.transAxes,
                 fontsize=7, color='yellow', alpha=0.9)

plt.tight_layout()
plt.savefig('variacion_parametros.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a12')
plt.show()
print('Figura guardada como variacion_parametros.png')


## Celda 7 — Simulacion cuantica: foton por foton

Esta celda muestra el aspecto mas profundo del experimento:
cuando se envian fotones **uno a uno**, el patron de interferencia
emerge estadisticamente, aunque cada foton solo active **un punto**.

Esto demuestra que **cada foton interfiere consigo mismo**.
La paradoja: la onda pasa por las dos rendijas, la particula solo por una.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 7 — Simulacion cuantica: llegada de fotones uno a uno
# ─────────────────────────────────────────────────────────────

np.random.seed(42)  # semilla para reproducibilidad

# Etapas: numero de fotones acumulados en cada panel
etapas_n = [50, 200, 1000, 5000, 20000]

# Distribucion de probabilidad teorica (patron de Fraunhofer)
I_teoria = intensidad_fraunhofer(y, A_MM, D_MM, LAMBDA_MM, L_MM, modo='doble')
I_prob   = I_teoria / I_teoria.sum()   # normalizar como distribucion

fig, axes = plt.subplots(2, 5, figsize=(18, 6),
                         gridspec_kw={'height_ratios': [1, 2]})
fig.suptitle('Emergencia del patron cuantico foton por foton', fontsize=12)

for col, n_fotones in enumerate(etapas_n):
    # Muestrear posiciones de impacto segun la distribucion teorica
    # Cada foton llega a una posicion aleatoria pero sesgada por I(y)
    indices    = np.random.choice(len(y), size=n_fotones, p=I_prob)
    posiciones = y[indices]

    # ── Imagen de acumulacion de impactos ──
    ax_img = axes[0][col]
    counts, _ = np.histogram(posiciones, bins=200,
                             range=(-Y_RANGO_MM, Y_RANGO_MM))
    counts_norm = counts / counts.max() if counts.max() > 0 else counts
    ax_img.imshow(np.tile(np.sqrt(counts_norm).reshape(1,-1), (50, 1)),
                  aspect='auto',
                  extent=[-Y_RANGO_MM, Y_RANGO_MM, 0, 1],
                  cmap=cmap_laser, vmin=0, vmax=1, origin='lower')
    ax_img.set_yticks([])
    ax_img.set_xticks([])
    ax_img.set_title(f'n = {n_fotones:,}\nfotones', fontsize=8)

    # ── Histograma + curva teorica superpuesta ──
    ax_curva = axes[1][col]
    ax_curva.hist(posiciones, bins=150,
                 range=(-Y_RANGO_MM, Y_RANGO_MM),
                 density=True, color=color_rgb, alpha=0.6,
                 label='Impactos' if col == 0 else None)

    # Curva teorica normalizada para superponerla
    I_norm = I_teoria / (I_teoria.sum() * (2*Y_RANGO_MM/len(y)))
    ax_curva.plot(y, I_norm, color='white', lw=0.8, alpha=0.6,
                 label='Teoria' if col == 0 else None)

    ax_curva.set_xlim(-Y_RANGO_MM, Y_RANGO_MM)
    ax_curva.set_xlabel('y [mm]', fontsize=8)
    if col == 0:
        ax_curva.set_ylabel('Densidad', fontsize=8)
        ax_curva.legend(fontsize=7)
    ax_curva.grid(True)
    ax_curva.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('simulacion_cuantica.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a12')
plt.show()
print('Figura guardada como simulacion_cuantica.png')
print()
print('Con pocos fotones el patron parece aleatorio.')
print('Con muchos fotones emerge el patron de interferencia.')
print('Esto es la dualidad onda-particula en accion.')


## Celda 8 — Resumen matematico

### Las 4 formulas clave

| Formula | Significado |
|---|---|
| $I(y) = I_0 \\cdot \\text{sinc}^2(\\alpha) \\cdot \\cos^2(\\beta)$ | Intensidad doble rendija |
| $\\alpha = \\frac{\\pi a \\sin\\theta}{\\lambda}$ | Parametro de difraccion |
| $\\beta = \\frac{\\pi d \\sin\\theta}{\\lambda}$ | Parametro de interferencia |
| $\\Delta y = \\frac{\\lambda L}{d}$ | Separacion entre franjas |

### Lo que observaste en el lab explicado matematicamente

- **Aumentar a** → alpha crece → sinc² se estrecha → lobulo central mas angosto
- **Aumentar d** → beta crece mas rapido → cos² oscila mas → mas franjas por angulo
- **Doble vs simple** → el factor cos²(beta) añade las franjas finas dentro del lobulo


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 8 — Tabla de resultados numericos
# ─────────────────────────────────────────────────────────────

print('=' * 65)
print('RESUMEN NUMERICO DEL EXPERIMENTO')
print('=' * 65)
print()

configs = [
    (0.04, 0.25, 100),
    (0.04, 0.50, 100),
    (0.08, 0.25, 100),
    (0.08, 0.50, 100),
]

lam_local = LAMBDA_NM * 1e-6
print(f'  Laser: lambda = {LAMBDA_NM} nm')
print()
print(f'{"a (mm)":>8} {"d (mm)":>8} {"L (cm)":>8} {"dy (mm)":>10} {"lobulo (mm)":>13}')
print('-' * 55)

for a, d, L in configs:
    L_mm_local  = L * 10
    delta_y_num = lam_local * L_mm_local / d
    lobulo_num  = 2 * lam_local * L_mm_local / a
    n_franjas   = int(lobulo_num / delta_y_num)
    print(f'{a:>8.2f} {d:>8.2f} {L:>8.0f} {delta_y_num:>10.2f} {lobulo_num:>13.1f}'
          f'   (~{n_franjas} franjas)')

print()
print('dy = separacion entre franjas de interferencia')
print('lobulo = ancho del lobulo central de difraccion')
print()
print('Conclusiones:')
print('  Aumentar d -> menor dy -> franjas mas juntas (mas visibles entre picos)')
print('  Aumentar a -> lobulo mas angosto -> picos mas estrechos')
